# SWE SFT data selection: 10K unique, high-quality tasks (v3)

Pipeline: **load → inspect raw schemas → normalize into one table → feature engineering → EDA → hard filters → quality scoring → dedup (exact + near-dup) → diversity-aware selection → save**.

Every step is its own cell so you can stop, look at the numbers, and tune the knobs in the **Config** cell.

### Changes in v3 (harness readiness)
- **SWE-smith**: its `patch` column *introduces* the bug. It is now kept as `setup_patch` (applied to the image before the agent starts) and `patch` is the **reverse**, i.e. the fix the agent must write. Rows whose bug patch can't be reversed (binary) are dropped by the existing "no non-test source changes" filter.
- **R2E-Gym**: `test_patch` is now empty (the commit's own test changes are kept in `commit_test_patch` for reference). R2E verifies with the hidden tests baked into its image (`/r2e_tests` + `run_tests.sh`, compared with `expected_output_json`), not with a test patch. `base_commit` is the parent of the fix commit (`<sha>^`), `fix_commit` is the commit itself.
- **Docker images**: derived for rows that have none (`IMAGE_RULES`): SWE-bench `swebench/sweb.eval.x86_64.<org>_1776_<repo>-<n>`, SWE-Gym `xingyaoww/sweb.eval.x86_64.<org>_s_<repo>-<n>`. SWE-bench Pro uses `docker_image` if present, else `jefzda/sweap-images:<dockerhub_tag>` (the tag alone is not pullable). `env_available` now means "has an image".
- **Harness fields kept**: `version`, `environment_setup_commit`, `verifier`, `env_image_source`, `setup_patch`, `fix_commit`, and `raw_json` (the full original row minus very large fields) so every source keeps what its test runner needs.
- **Mixed-up rows**: a new filter (`DROP_UNLINKED_F2P`) drops rows whose FAIL_TO_PASS tests share no name with their `test_patch`, which is what a row with another task's problem statement and patch looks like.
- **Integrity check** (section 12.1): every selected row is rebuilt from the original dataset row and compared field by field; the notebook stops before saving if anything is misaligned.
- **Saving**: Drive is mounted first, primary and reserve are both saved, and `df` is no longer overwritten by the reserve set.

### Changes in v2
- **Source caps** now apply to the 10K primary set: primary and reserve are selected separately, and when slots are short the repo cap is lifted before the source cap (section 11).
- **Repo names**: R2E-Gym bare names (`pandas`, `numpy`, `sympy`, ...) map to `owner/repo` via `REPO_ALIASES`; section 6.1 lists any left over.
- **ProMax hints** are dropped (`DROP_HINTS_SOURCES`), since they summarize the gold patch; no hint bonus for them. `hints` is never meant for the agent prompt.
- **Scoring**: LLM-written issues (SWE-smith, R2E-Gym) get `SYNTHETIC_PS_DISCOUNT` on the problem-statement score; a continuous tie-breaker spreads out tied scores; score is capped at 100.
- **Problem statements** are cleaned of issue-template HTML comments and extra blank lines.
- **Language labels** are normalized (`js` -> `javascript`, `ts` -> `typescript`, `c++` -> `cpp`).
- **Near-dup threshold** lowered from 0.85 to 0.70 (check the examples in section 10).


## 0. Setup

In [ ]:
!pip -q install -U datasets datasketch pyarrow matplotlib tqdm

In [ ]:
# If any dataset is gated, log in first (SWE-bench Pro / ProMax may require accepting terms on the Hub)
# from huggingface_hub import notebook_login; notebook_login()

In [ ]:
import re, json, ast, hashlib, difflib, math, random, textwrap
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
random.seed(0); np.random.seed(0)

## 1. Config (tune these)

In [ ]:
TARGET_N   = 10_000      # primary set you will generate trajectories for
RESERVE_N  = 5_000       # ranked extra tasks: not every trajectory will succeed, so keep a backup queue

# ---- hard filters ----
MIN_PS_CHARS       = 80      # problem statement too short = underspecified
MAX_PS_TOKENS      = 6_000   # rough (chars/4); very long issues eat Gemma's context
MAX_SRC_FILES      = 15      # gold patch touches too many files -> very long trajectories
MAX_SRC_LINES      = 1_500   # added+removed lines in non-test files
MAX_LEAK_RATIO     = 0.6     # fraction of gold-patch lines that appear verbatim in the issue
REQUIRE_ENV        = True    # you must be able to run the repo to generate/verify trajectories
REQUIRE_TESTS      = True    # need tests to verify that a trajectory actually solved the task
DROP_UNLINKED_F2P  = True    # drop rows whose FAIL_TO_PASS tests have nothing to do with their test_patch
LANGUAGES          = None    # e.g. {"python"} to keep Python only; None keeps everything

# ---- dedup ----
NEAR_DUP_THRESHOLD = 0.70    # (was 0.85, caught only 10) Jaccard on 5-gram word shingles of the problem statement
NUM_PERM           = 128

# ---- diversity ----
MAX_PER_REPO       = 250     # cap per canonical repo (avoids django / one smith repo dominating)
SOURCE_MAX_FRAC    = {       # max share of the final set per source (others uncapped)
    "swe_smith": 0.45,
    "swe_bench": 0.35,
    "r2e_gym":   0.40,
}
TAKE_ALL_SOURCES   = ["swe_bench_verified", "swe_bench_pro", "swe_bench_promax", "swe_gym"]  # small + high value: include every survivor

# ---- source prior (how much you trust each source a priori, 0..1) ----
SOURCE_PRIOR = {
    "swe_bench_verified": 1.00,   # human-validated issues + tests
    "swe_bench_pro":      0.95,   # curated, long-horizon, multi-language
    "swe_bench_promax":   0.90,   # curated refactoring, but large patches
    "swe_gym":            0.85,   # real issues, executable envs
    ("swe_bench", "test"): 0.80,
    ("swe_bench", "dev"):  0.80,
    ("swe_bench", "train"):0.50,  # real issues but (usually) no validated tests / prebuilt envs
    "r2e_gym":            0.75,   # real commits, LLM-written issues
    "swe_smith":          0.60,   # synthetic bugs, LLM-written issues
}

# ---- scoring fixes ----
SYNTHETIC_ISSUE_SOURCES = {"swe_smith", "r2e_gym"}   # LLM-written issues follow the template the regex signals reward
SYNTHETIC_PS_DISCOUNT   = 0.75                        # multiply their problem-statement score by this
DROP_HINTS_SOURCES      = {"swe_bench_promax"}        # ProMax hints_text is an LLM summary of the gold patch (leak)

# ---- environments ----
# Rows that carry no image column get one derived from instance_id with these rules.
# (template, separator that replaces "__" in the instance id). Anything not listed and without
# an image column has no environment (e.g. SWE-bench train) and is removed by REQUIRE_ENV.
IMAGE_RULES = {
    "swe_bench_verified":  ("swebench/sweb.eval.x86_64.{key}:latest", "_1776_"),
    ("swe_bench", "test"): ("swebench/sweb.eval.x86_64.{key}:latest", "_1776_"),
    ("swe_bench", "dev"):  ("swebench/sweb.eval.x86_64.{key}:latest", "_1776_"),  # check these exist before relying on them
    "swe_gym":             ("xingyaoww/sweb.eval.x86_64.{key}:latest", "_s_"),
}
PRO_V1_IMAGE_REPO = "jefzda/sweap-images"   # SWE-bench Pro v1: full image = f"{PRO_V1_IMAGE_REPO}:{dockerhub_tag}"
IMAGE_COLUMNS = ["docker_image", "image_name", "image", "env_image"]  # full image references, in priority order

# How each source is verified downstream (the trajectory pipeline routes on this column).
VERIFIER = {
    "swe_bench_verified": "swebench",      # SWE-bench test spec selected by repo + version
    "swe_bench":          "swebench",
    "swe_gym":            "swe_gym",       # SWE-bench-style FAIL_TO_PASS / PASS_TO_PASS with pytest
    "swe_smith":          "swe_smith",     # apply setup_patch (the bug) first; F2P/P2P are existing tests
    "r2e_gym":            "r2e_gym",       # hidden tests in the image, compared with expected_output_json
    "swe_bench_pro":      "swe_bench_pro", # per-instance run script from scaleapi/SWE-bench_Pro-os
    "swe_bench_promax":   "promax",        # every test must pass; check its columns in section 3
}

# Original-row fields too large to keep in raw_json (R2E-Gym stores full file contents).
RAW_DROP_KEYS       = {"parsed_commit_content", "execution_result_content"}
RAW_MAX_FIELD_CHARS = 200_000

def lookup(d, source, split, default=None):
    return d.get((source, split), d.get(source, default))

## 2. Load all datasets

In [ ]:
DATASET_IDS = {
    "swe_gym":            "SWE-Gym/SWE-Gym",
    "swe_bench_verified": "SWE-bench/SWE-bench_Verified",
    "swe_bench":          "SWE-bench/SWE-bench",
    "r2e_gym":            "R2E-Gym/R2E-Gym-V1",
    "swe_smith":          "SWE-bench/SWE-smith",
    "swe_bench_pro":      "ScaleAI/SWE-bench_Pro",
    "swe_bench_promax":   "swe-bench-promax/SWE-Bench-ProMax",
}

raw = {}
for name, hf_id in DATASET_IDS.items():
    try:
        raw[name] = load_dataset(hf_id)
        print(f"OK     {name:20s} {hf_id:40s} splits={ {k: len(v) for k, v in raw[name].items()} }")
    except Exception as e:
        print(f"FAILED {name:20s} {hf_id:40s} -> {type(e).__name__}: {str(e)[:200]}")

## 3. Investigate raw data
### 3.1 Schemas: splits, sizes, columns

In [ ]:
for name, dd in raw.items():
    print("=" * 110)
    print(f"{name}  ({DATASET_IDS[name]})")
    for split, ds in dd.items():
        print(f"  split={split:<8s} rows={len(ds):>8,}")
    first_split = next(iter(dd))
    for col, feat in dd[first_split].features.items():
        print(f"      - {col:<32s} {str(feat)[:70]}")

### 3.2 Which standard fields does each dataset have?

In [ ]:
STANDARD_FIELDS = {
    "instance_id":        ["instance_id"],
    "repo":               ["repo", "repo_name"],
    "base_commit":        ["base_commit", "commit_hash"],
    "problem_statement":  ["problem_statement"],
    "hints":              ["hints_text"],
    "patch":              ["patch", "parsed_commit_content"],
    "test_patch":         ["test_patch"],
    "FAIL_TO_PASS":       ["FAIL_TO_PASS", "fail_to_pass", "expected_output_json"],
    "PASS_TO_PASS":       ["PASS_TO_PASS", "pass_to_pass"],
    "env/image":          IMAGE_COLUMNS + ["dockerhub_tag"],
    "version":            ["version"],
    "env_setup_commit":   ["environment_setup_commit"],
    "language":           ["language", "repo_language"],
    "created_at":         ["created_at"],
}
rows = []
for name, dd in raw.items():
    for split, ds in dd.items():
        cols = set(ds.column_names)
        r = {"dataset": name, "split": split, "rows": len(ds)}
        for f, cands in STANDARD_FIELDS.items():
            hit = [c for c in cands if c in cols]
            r[f] = hit[0] if hit else "-"
        rows.append(r)
coverage = pd.DataFrame(rows)
coverage

### 3.3 Peek at one example from any dataset

In [ ]:
def show_example(name, split=None, idx=None, max_chars=700):
    dd = raw[name]
    split = split or next(iter(dd))
    ds = dd[split]
    idx = random.randrange(len(ds)) if idx is None else idx
    ex = ds[idx]
    print(f"### {name} / {split} / row {idx}")
    for k, v in ex.items():
        s = v if isinstance(v, str) else json.dumps(v, default=str)
        s = s if len(s) <= max_chars else s[:max_chars] + f" ... [+{len(s) - max_chars:,} chars]"
        print(f"\n--- {k} ---\n{s}")

show_example("swe_bench_verified")

In [ ]:
if "swe_smith" in raw:
    show_example("swe_smith")

In [ ]:
if "r2e_gym" in raw:
    show_example("r2e_gym", max_chars=400)

### 3.4 R2E-Gym stores the commit as JSON, not a unified diff
Check the structure so the patch reconstruction below finds the file paths correctly.

In [ ]:
if "r2e_gym" in raw:
    _r2e = raw["r2e_gym"][next(iter(raw["r2e_gym"]))]
    _obj = json.loads(_r2e[0]["parsed_commit_content"])
    print("top-level keys:", list(_obj.keys()))
    _fd = _obj["file_diffs"][0]
    print("file_diff keys:", list(_fd.keys()))
    print("header:", json.dumps(_fd.get("header"), indent=1)[:800])
    print("\nexpected_output_json sample:", _r2e[0]["expected_output_json"][:300])

## 4. Helpers: parsing lists, patches, languages, repos

In [ ]:
TEST_PATH_RE = re.compile(
    r"(^|/)(tests?|testing|__tests__|spec|specs)(/|$)"
    r"|(^|/)test_[^/]*$|_tests?\.[a-z]+$|\.(test|spec)\.[a-z]+$"
    r"|Tests?\.(java|kt|cs|scala)$|(^|/)conftest\.py$",
    re.I,
)
EXT_LANG = {
    ".py": "python", ".pyx": "python", ".go": "go", ".js": "javascript", ".jsx": "javascript",
    ".mjs": "javascript", ".ts": "typescript", ".tsx": "typescript", ".java": "java",
    ".rs": "rust", ".c": "c", ".h": "c", ".cc": "cpp", ".cpp": "cpp", ".hpp": "cpp",
    ".rb": "ruby", ".php": "php", ".cs": "csharp", ".kt": "kotlin", ".scala": "scala", ".swift": "swift",
}
CODE_EXTS = set(EXT_LANG)

def parse_list(x):
    """FAIL_TO_PASS is a JSON string in SWE-bench, a python-repr string in some sets, a list in others."""
    if x is None:
        return None
    if isinstance(x, (list, tuple, np.ndarray)):
        return [str(i) for i in x]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        for loader in (json.loads, ast.literal_eval):
            try:
                v = loader(s)
                if isinstance(v, (list, tuple)):
                    return [str(i) for i in v]
            except Exception:
                pass
        return [s]
    return None

def first_present(r, keys, default=None):
    for k in keys:
        v = r.get(k)
        if v is None or (isinstance(v, str) and not v.strip()):
            continue
        return v
    return default

def split_patch_files(patch):
    if not patch:
        return []
    out = []
    for chunk in re.split(r"(?m)^(?=diff --git )", patch):
        if not chunk.startswith("diff --git"):
            continue
        m = re.match(r"diff --git a/(\S+) b/(\S+)", chunk)
        out.append((m.group(2) if m else "?", chunk))
    return out

def split_patch_by_tests(patch):
    src, tst = [], []
    for path, chunk in split_patch_files(patch):
        (tst if TEST_PATH_RE.search(path) else src).append(chunk)
    return "".join(src), "".join(tst)

def patch_stats(patch):
    files = split_patch_files(patch)
    st = dict(n_files=len(files), n_src_files=0, n_test_files=0, n_code_files=0,
              src_added=0, src_removed=0, n_hunks=0, src_paths=[])
    for path, chunk in files:
        is_test = bool(TEST_PATH_RE.search(path))
        if is_test:
            st["n_test_files"] += 1
            continue
        st["n_src_files"] += 1
        st["src_paths"].append(path)
        ext = "." + path.rsplit(".", 1)[-1].lower() if "." in path else ""
        st["n_code_files"] += ext in CODE_EXTS
        for line in chunk.splitlines():
            if line.startswith(("+++", "---")):
                continue
            if line.startswith("@@"):
                st["n_hunks"] += 1
            elif line.startswith("+"):
                st["src_added"] += 1
            elif line.startswith("-"):
                st["src_removed"] += 1
    st["src_lines"] = st["src_added"] + st["src_removed"]
    return st

def lang_from_paths(paths):
    c = Counter()
    for p in paths:
        if "." in p:
            l = EXT_LANG.get("." + p.rsplit(".", 1)[-1].lower())
            if l:
                c[l] += 1
    return c.most_common(1)[0][0] if c else "unknown"

def normalized_patch_hash(patch):
    """Hash of changed lines only (ignores index lines, line numbers, whitespace)."""
    if not patch:
        return None
    keep = []
    for path, chunk in split_patch_files(patch):
        keep.append("F:" + path)
        for line in chunk.splitlines():
            if line.startswith(("+++", "---")):
                continue
            if line and line[0] in "+-":
                keep.append(line[0] + re.sub(r"\s+", "", line[1:]))
    return hashlib.sha1("\n".join(keep).encode()).hexdigest() if keep else None

def text_hash(s):
    s = re.sub(r"\s+", " ", (s or "").lower()).strip()
    return hashlib.sha1(s.encode()).hexdigest() if s else None

# R2E-Gym's repo_name has no owner ("pandas"), so it would get its own repo cap next to "pandas-dev/pandas".
# Map bare names to owner/repo. Section 6.1 prints any bare name still left so you can extend this.
REPO_ALIASES = {
    "pandas": "pandas-dev/pandas", "numpy": "numpy/numpy", "sympy": "sympy/sympy",
    "orange3": "biolab/orange3", "aiohttp": "aio-libs/aiohttp", "pillow": "python-pillow/pillow",
    "tornado": "tornadoweb/tornado", "scrapy": "scrapy/scrapy", "datalad": "datalad/datalad",
    "pyramid": "pylons/pyramid", "coveragepy": "nedbat/coveragepy", "bokeh": "bokeh/bokeh",
}

LANG_ALIASES = {"js": "javascript", "ts": "typescript", "c++": "cpp", "golang": "go", "py": "python", "python3": "python"}

def canon_lang(lang):
    l = str(lang).strip().lower()
    return LANG_ALIASES.get(l, l)

HTML_COMMENT_RE = re.compile(r"<!--.*?-->", re.S)

def clean_problem_statement(s):
    """Drop GitHub issue-template HTML comments and collapse blank lines (saves context tokens)."""
    if not s:
        return ""
    s = HTML_COMMENT_RE.sub("", str(s)).replace("\r\n", "\n")
    s = re.sub(r"[ \t]+\n", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def canon_repo(repo):
    """'swesmith/pandas-dev__pandas.95280573' -> 'pandas-dev/pandas'; 'django/django' -> 'django/django'."""
    if not repo:
        return "unknown"
    r = str(repo).strip().lower()
    r = re.sub(r"^swesmith/", "", r)
    r = re.sub(r"\.[0-9a-f]{7,40}$", "", r)
    r = r.replace("__", "/")
    return REPO_ALIASES.get(r, r)

# ---- R2E-Gym: rebuild a unified diff from old/new file contents ----
def _find_path(obj, depth=0):
    if depth > 4 or not isinstance(obj, dict):
        return None
    for k in ("path", "file_path", "filename", "new_path", "old_path"):
        v = obj.get(k)
        if isinstance(v, str) and v:
            return v
    for v in obj.values():
        p = _find_path(v, depth + 1)
        if p:
            return p
    return None

def r2e_build_patch(parsed_commit_content, max_chars=3_000_000):
    try:
        obj = json.loads(parsed_commit_content)
    except Exception:
        return ""
    parts = []
    for i, fd in enumerate(obj.get("file_diffs", []) if isinstance(obj, dict) else []):
        path = _find_path(fd.get("header") or {}) or _find_path(
            {k: v for k, v in fd.items() if "content" not in k}) or f"unknown_{i}"
        old, new = fd.get("old_file_content") or "", fd.get("new_file_content") or ""
        if len(old) + len(new) > max_chars:
            continue
        body = "\n".join(difflib.unified_diff(old.splitlines(), new.splitlines(),
                                              f"a/{path}", f"b/{path}", lineterm="", n=3))
        if body:
            parts.append(f"diff --git a/{path} b/{path}\n{body}\n")
    return "".join(parts)

# quick sanity check
_p = "diff --git a/src/x.py b/src/x.py\n--- a/src/x.py\n+++ b/src/x.py\n@@ -1 +1 @@\n-a=1\n+a=2\n"
print(patch_stats(_p), canon_repo("swesmith/pandas-dev__pandas.95280573"), canon_repo("pandas"), canon_lang("js"))
print(repr(clean_problem_statement("Bug <!-- template\njunk -->\n\n\n\nDetails  \n")))

# ---- SWE-smith: its patch introduces the bug; the fix is the reverse diff ----
_HUNK_HDR = re.compile(r"^@@ -(\d+(?:,\d+)?) \+(\d+(?:,\d+)?) @@(.*)$")


def reverse_patch(patch):
    """Invert a unified git diff (what `git apply -R` would apply). Returns None if it can't.

    SWE-smith's `patch` *introduces* the bug; the fix the agent must write is its reverse.
    Binary diffs cannot be inverted from text, so they return None.
    """
    if not patch or not patch.strip():
        return None
    lines = patch.split("\n")
    trailing_nl = patch.endswith("\n")
    if trailing_nl:
        lines = lines[:-1]
    out, i, n = [], 0, len(lines)
    while i < n:
        line = lines[i]
        if line.startswith("GIT binary patch") or (line.startswith("Binary files ") and line.endswith(" differ")):
            return None
        if line.startswith("diff --git "):
            m = re.match(r'^diff --git (\S+) (\S+)$', line)
            if m:
                a, b = m.group(1), m.group(2)
                a2 = "a/" + b[2:] if b.startswith("b/") else b
                b2 = "b/" + a[2:] if a.startswith("a/") else a
                out.append(f"diff --git {a2} {b2}")
            else:
                out.append(line)
            i += 1
            continue
        if line.startswith("--- ") and i + 1 < n and lines[i + 1].startswith("+++ "):
            old, new = line[4:], lines[i + 1][4:]
            new_as_old = "a/" + new[2:] if new.startswith("b/") else new
            old_as_new = "b/" + old[2:] if old.startswith("a/") else old
            out += ["--- " + new_as_old, "+++ " + old_as_new]
            i += 2
            continue
        swaps = (("new file mode ", "deleted file mode "), ("deleted file mode ", "new file mode "),
                 ("old mode ", "new mode "), ("new mode ", "old mode "),
                 ("rename from ", "rename to "), ("rename to ", "rename from "),
                 ("copy from ", "copy to "), ("copy to ", "copy from "))
        swapped = False
        for pre, rep in swaps:
            if line.startswith(pre):
                out.append(rep + line[len(pre):])
                swapped = True
                break
        if swapped:
            i += 1
            continue
        if line.startswith("index "):
            m = re.match(r"^index ([0-9a-f]+)\.\.([0-9a-f]+)(.*)$", line)
            out.append(f"index {m.group(2)}..{m.group(1)}{m.group(3)}" if m else line)
            i += 1
            continue
        m = _HUNK_HDR.match(line)
        if m:
            old_r, new_r, ctx = m.groups()
            out.append(f"@@ -{new_r} +{old_r} @@{ctx}")

            def _count(r):
                return int(r.split(",")[1]) if "," in r else 1

            rem_old, rem_new = _count(old_r), _count(new_r)
            i += 1
            block = []  # pending change lines: (sign, text, no_newline_marker)

            def flush():
                minus = [x for x in block if x[0] == "-"]
                plus = [x for x in block if x[0] == "+"]
                for sign, text, marker in minus + plus:
                    out.append(sign + text)
                    if marker:
                        out.append(marker)
                block.clear()

            while i < n and (rem_old > 0 or rem_new > 0 or (lines[i].startswith("\\"))):
                hl = lines[i]
                if hl.startswith("\\"):
                    if block:
                        s, t, _ = block[-1]
                        block[-1] = (s, t, hl)
                    else:
                        out.append(hl)
                    i += 1
                    continue
                tag, text = (hl[:1], hl[1:]) if hl else (" ", "")
                if tag == " ":
                    flush()
                    out.append(" " + text)
                    rem_old -= 1
                    rem_new -= 1
                elif tag == "-":
                    block.append(("+", text, None))  # removed in bug patch -> added by the fix
                    rem_old -= 1
                elif tag == "+":
                    block.append(("-", text, None))
                    rem_new -= 1
                else:
                    return None  # malformed hunk
                i += 1
            flush()
            continue
        out.append(line)
        i += 1
    return "\n".join(out) + ("\n" if trailing_nl else "")


_F2P_STOP = {"test", "tests", "self", "should", "with", "when", "that", "true", "false", "none", "returns", "return"}

def f2p_in_test_patch(f2p, test_patch):
    """False when no FAIL_TO_PASS entry shares a test name, module or file name with the test_patch.

    That happens when a row's problem statement / patch / test_patch were copied from another task
    (e.g. a django instance_id with an xarray patch). Rows without F2P or test_patch are not judged.
    """
    if not isinstance(f2p, list) or not f2p or not test_patch:
        return True
    for t in f2p:
        for tok in re.findall(r"[A-Za-z_][A-Za-z0-9_]{3,}", str(t)):
            if tok.lower() not in _F2P_STOP and tok in test_patch:
                return True
    return False

assert f2p_in_test_patch(["test_union (queries.test_qs_combinators.QuerySetSetOperationTests)"],
                         "diff --git a/xarray/tests/test_dataarray.py ...") is False
assert f2p_in_test_patch(["TestServer/handles_empty"], "+func TestServer(t *testing.T) {") is True

def original_row(rec):
    """The untouched dataset row for a normalized record (needs `raw` from section 2)."""
    return raw[rec["source"]][rec["split"]][int(rec["row_idx"])]

_bug = "diff --git a/x.py b/x.py\n--- a/x.py\n+++ b/x.py\n@@ -1,2 +1,2 @@\n a = 1\n-b = 2\n+b = 3\n"
assert reverse_patch(_bug) == "diff --git a/x.py b/x.py\n--- a/x.py\n+++ b/x.py\n@@ -1,2 +1,2 @@\n a = 1\n-b = 3\n+b = 2\n"


## 5. Normalize everything into one table
One row per (source, split, row). `row_idx` lets you go back to the original row later to get the exact fields your trajectory harness needs.

Notes:
- **SWE-smith**: the dataset's `patch` *introduces* the bug. It becomes `setup_patch` (applied to the image before the agent starts) and `patch` is its reverse: the fix. There is no `test_patch`; FAIL_TO_PASS are existing tests the bug breaks.
- **R2E-Gym**: issues are LLM-written from commits; the patch is rebuilt from file contents. Its test part is kept as `commit_test_patch` only: R2E verifies with hidden tests inside the image and `expected_output_json`, so `test_patch` is empty and there is no FAIL_TO_PASS list. `base_commit = <fix commit>^`.
- **SWE-bench Pro**: `requirements` and `interface` are kept in `extra_context`; give them to the agent too.
- Every row gets `env_image` (from the row or `IMAGE_RULES`), `verifier`, `version` and `environment_setup_commit`.

In [ ]:
def resolve_image(source, split, instance_id, r):
    """(image, how) for a row: an image column, SWE-bench Pro's tag, or a naming rule."""
    img = first_present(r, IMAGE_COLUMNS)
    if img:
        return str(img).strip(), "row"
    tag = first_present(r, ["dockerhub_tag"])
    if tag:
        tag = str(tag).strip()
        return (tag if "/" in tag else f"{PRO_V1_IMAGE_REPO}:{tag}"), "row_tag"
    rule = lookup(IMAGE_RULES, source, split)
    if rule and instance_id:
        template, sep = rule
        return template.format(key=str(instance_id).replace("__", sep).lower()), "derived"
    return None, None


def normalize_row(source, split, i, r):
    rec = dict(source=source, split=split, row_idx=i, verifier=VERIFIER.get(source, "unknown"),
               setup_patch="", fix_commit="", commit_test_patch="")
    if source == "r2e_gym":
        repo, commit = r.get("repo_name"), r.get("commit_hash")
        full = r2e_build_patch(r.get("parsed_commit_content"))
        patch, test_part = split_patch_by_tests(full)
        ps = clean_problem_statement(re.sub(r"^\s*\[ISSUE\]\s*|\s*\[/ISSUE\]\s*$", "", r.get("problem_statement") or ""))
        try:
            n_tests = len(json.loads(r.get("expected_output_json") or "{}"))
        except Exception:
            n_tests = None
        image, how = resolve_image(source, split, None, r)
        rec.update(instance_id=f"{repo}__{commit}", repo=repo,
                   base_commit=f"{commit}^" if commit else None,   # the image's /testbed is the parent of the fix commit
                   fix_commit=commit or "",
                   problem_statement=ps, extra_context="", hints="",
                   patch=patch, test_patch="",                      # R2E verifies with hidden tests in the image
                   commit_test_patch=test_part,
                   f2p=None, p2p=None, n_tests=n_tests,
                   env_image=image, env_image_source=how, version=None, environment_setup_commit=None,
                   language_raw="python", created_at=None)
        return rec

    extra = ""
    if r.get("requirements"):
        extra += "## Requirements\n" + str(r["requirements"]).strip() + "\n\n"
    if r.get("interface"):
        extra += "## Interface\n" + str(r["interface"]).strip() + "\n"
    f2p = parse_list(first_present(r, ["FAIL_TO_PASS", "fail_to_pass"]))
    if f2p is None and ("FAIL_TO_PASS" in r or "fail_to_pass" in r):
        f2p = []
    instance_id = first_present(r, ["instance_id"], f"{source}-{split}-{i}")
    patch = first_present(r, ["patch"], "") or ""
    setup_patch = ""
    if source == "swe_smith":
        # SWE-smith's patch introduces the bug: apply it during setup, train on its reverse.
        setup_patch = patch
        patch = reverse_patch(patch) or ""
    image, how = resolve_image(source, split, instance_id, r)
    version = first_present(r, ["version"])
    rec.update(
        instance_id=instance_id,
        repo=first_present(r, ["repo", "repo_name"]),
        base_commit=first_present(r, ["base_commit", "commit_hash"]),
        problem_statement=clean_problem_statement(first_present(r, ["problem_statement"], "")),
        extra_context=extra,
        # hints are kept for reference only: never show them to the agent. ProMax hints describe the gold patch.
        hints="" if source in DROP_HINTS_SOURCES else (first_present(r, ["hints_text"], "") or ""),
        patch=patch,
        setup_patch=setup_patch,
        test_patch=first_present(r, ["test_patch"], "") or "",
        f2p=f2p,
        p2p=parse_list(first_present(r, ["PASS_TO_PASS", "pass_to_pass"])),
        n_tests=None,
        env_image=image,
        env_image_source=how,
        version=str(version) if version is not None else None,
        environment_setup_commit=first_present(r, ["environment_setup_commit"]),
        language_raw=first_present(r, ["language", "repo_language"]),
        created_at=str(r.get("created_at")) if r.get("created_at") is not None else None,
    )
    return rec

records = []
for source, dd in raw.items():
    for split, ds in dd.items():
        for i, r in enumerate(tqdm(ds, desc=f"{source}/{split}", leave=False)):
            records.append(normalize_row(source, split, i, r))

df = pd.DataFrame(records)
df["uid"] = df["source"] + "::" + df["split"] + "::" + df["row_idx"].astype(str)
print(f"{len(df):,} rows total")
smith = df["source"] == "swe_smith"
if smith.any():
    print(f"SWE-smith bug patches that could not be reversed (dropped later): {(smith & (df['patch'] == '')).sum():,}")
display(pd.crosstab([df["source"], df["split"]], df["env_image_source"].fillna("none")))
df.groupby(["source", "split"]).size().rename("rows").to_frame()

## 6. Feature engineering

In [ ]:
CODE_BLOCK_RE = re.compile(r"```|^(?: {4}|\t)\S", re.M)
TRACE_RE      = re.compile(r"Traceback \(most recent call last\)|\b\w+(?:Error|Exception)\b|panic:|\bat [\w.$]+\(", re.I)
REPRO_RE      = re.compile(r"reproduc|steps to|minimal (?:example|code)|mcve|example:|to replicate", re.I)
EXPECT_RE     = re.compile(r"expected (?:behaviou?r|result|output)|actual (?:behaviou?r|result|output)|should (?:be|return|raise|not)|instead of", re.I)
URL_RE        = re.compile(r"https?://\S+")

def leak_ratio(ps, patch):
    """Fraction of substantive added lines of the gold patch that appear verbatim in the issue."""
    added = [l[1:].strip() for _, c in split_patch_files(patch) for l in c.splitlines()
             if l.startswith("+") and not l.startswith("+++")]
    added = [a for a in added if len(a) >= 25]
    if not added or not ps:
        return 0.0
    return sum(a in ps for a in added) / len(added)

def add_features(df):
    st = df["patch"].map(patch_stats).apply(pd.Series)
    for c in st.columns:
        df[c] = st[c]
    ps = df["problem_statement"].fillna("")
    df["ps_chars"]       = ps.str.len()
    df["ps_words"]       = ps.str.split().str.len()
    df["ps_est_tokens"]  = (df["ps_chars"] + df["extra_context"].fillna("").str.len()) / 4
    df["ps_has_code"]    = ps.str.contains(CODE_BLOCK_RE)
    df["ps_has_trace"]   = ps.str.contains(TRACE_RE)
    df["ps_has_repro"]   = ps.str.contains(REPRO_RE)
    df["ps_has_expect"]  = ps.str.contains(EXPECT_RE)
    df["ps_url_frac"]    = ps.map(lambda s: sum(len(u) for u in URL_RE.findall(s)) / max(len(s), 1))
    df["ps_nonascii"]    = ps.map(lambda s: sum(ord(ch) > 127 for ch in s) / max(len(s), 1))
    df["ps_has_diff"]    = ps.str.contains(r"diff --git|^@@ ", regex=True)
    df["has_hints"]      = df["hints"].fillna("").str.len() > 20
    df["leak_ratio"]     = [leak_ratio(a, b) for a, b in tqdm(zip(ps, df["patch"]), total=len(df), desc="leak")]
    df["n_f2p"]          = df["f2p"].map(lambda x: len(x) if isinstance(x, list) else np.nan)
    df["n_p2p"]          = df["p2p"].map(lambda x: len(x) if isinstance(x, list) else np.nan)

    def _has_tests(r):
        if isinstance(r["f2p"], list):
            return len(r["f2p"]) > 0
        if pd.notna(r["n_tests"]):
            return r["n_tests"] > 0
        return bool(r["test_patch"])
    df["has_tests"] = df.apply(_has_tests, axis=1)
    df["f2p_linked"] = [f2p_in_test_patch(a, b) for a, b in zip(df["f2p"], df["test_patch"])]
    df["env_available"] = df["env_image"].fillna("").astype(str).str.len() > 0   # image from the row or IMAGE_RULES
    df["language"] = [canon_lang(l) if isinstance(l, str) and l.strip() else lang_from_paths(p)
                      for l, p in zip(df["language_raw"], df["src_paths"])]
    df["repo_canon"] = df["repo"].map(canon_repo)
    df["patch_hash"] = df["patch"].map(normalized_patch_hash)
    df["ps_hash"]    = df["problem_statement"].map(text_hash)
    return df

df = add_features(df)
df.head(3)

### 6.1 Check repo and language normalization

In [ ]:
bare = df.loc[~df["repo_canon"].str.contains("/"), ["source", "repo", "repo_canon"]].value_counts()
print("repos still without an owner (add them to REPO_ALIASES and re-run sections 4-6):")
display(bare.head(30) if len(bare) else "none")
print("languages:", {k: int(v) for k, v in df["language"].value_counts().items()})


## 7. EDA on the full pool
### 7.1 Per-source summary

In [ ]:
summary = df.groupby("source").agg(
    rows=("uid", "size"),
    repos=("repo_canon", "nunique"),
    pct_env=("env_available", "mean"),
    pct_tests=("has_tests", "mean"),
    med_ps_chars=("ps_chars", "median"),
    pct_empty_ps=("ps_chars", lambda s: (s < MIN_PS_CHARS).mean()),
    pct_code=("ps_has_code", "mean"),
    pct_trace=("ps_has_trace", "mean"),
    pct_repro=("ps_has_repro", "mean"),
    med_src_files=("n_src_files", "median"),
    med_src_lines=("src_lines", "median"),
    p90_src_lines=("src_lines", lambda s: s.quantile(0.9)),
    med_f2p=("n_f2p", "median"),
    pct_leak=("leak_ratio", lambda s: (s > MAX_LEAK_RATIO).mean()),
)
pct_cols = [c for c in summary.columns if c.startswith("pct_")]
summary.style.format({c: "{:.1%}" for c in pct_cols}).format(precision=0, subset=[c for c in summary.columns if c not in pct_cols])

### 7.2 Distributions

In [ ]:
def hist_by_source(col, data=None, logx=True, bins=40, title=None, clip_min=1):
    data = df if data is None else data
    vals = data[col].dropna().astype(float)
    if logx:
        vals = vals[vals >= clip_min]
        edges = np.logspace(np.log10(max(vals.min(), clip_min)), np.log10(vals.max() + 1), bins)
    else:
        edges = np.linspace(vals.min(), vals.max(), bins)
    fig, ax = plt.subplots(figsize=(12, 4))
    for s in sorted(data["source"].unique()):
        v = data.loc[data["source"] == s, col].dropna().astype(float)
        v = v[v >= clip_min] if logx else v
        if len(v):
            ax.hist(v, bins=edges, histtype="step", density=True, linewidth=1.6, label=f"{s} (n={len(v):,})")
    if logx:
        ax.set_xscale("log")
    ax.set_title(title or col); ax.set_ylabel("density"); ax.legend(fontsize=8, ncol=2)
    plt.show()

hist_by_source("ps_chars", title="Problem statement length (chars)")
hist_by_source("src_lines", title="Gold patch size: added+removed lines in non-test files")
hist_by_source("n_src_files", title="Gold patch: number of non-test files touched")
hist_by_source("n_f2p", title="Number of FAIL_TO_PASS tests")

### 7.3 Languages and repos

In [ ]:
lang_tab = pd.crosstab(df["language"], df["source"])
display(lang_tab.loc[lang_tab.sum(1).sort_values(ascending=False).index])

top_repos = df["repo_canon"].value_counts().head(30)
fig, ax = plt.subplots(figsize=(12, 6))
pd.crosstab(df["repo_canon"], df["source"]).loc[top_repos.index[::-1]].plot.barh(stacked=True, ax=ax)
ax.set_title("Top-30 repos across all sources"); plt.tight_layout(); plt.show()
print(f"unique canonical repos: {df['repo_canon'].nunique():,}")

### 7.4 Overlap between sources (contamination / duplicates)

In [ ]:
def overlap_matrix(key):
    sets = {s: set(g[key].dropna()) for s, g in df.groupby("source")}
    names = sorted(sets)
    return pd.DataFrame([[len(sets[a] & sets[b]) for b in names] for a in names], index=names, columns=names)

for key in ["instance_id", "patch_hash", "ps_hash"]:
    print(f"\n=== shared {key} ===")
    display(overlap_matrix(key))

print("within-source exact duplicates:")
display(df.groupby("source").agg(
    dup_instance=("instance_id", lambda s: s.duplicated().sum()),
    dup_patch=("patch_hash", lambda s: s.dropna().duplicated().sum()),
    dup_ps=("ps_hash", lambda s: s.dropna().duplicated().sum()),
))

### 7.5 Problem-statement quality flags by source

In [ ]:
flags = ["ps_has_code", "ps_has_trace", "ps_has_repro", "ps_has_expect", "has_hints", "ps_has_diff"]
ax = df.groupby("source")[flags].mean().plot.bar(figsize=(12, 4), rot=0)
ax.set_title("Share of problem statements with each signal"); ax.legend(fontsize=8, ncol=3); plt.show()

ax = df.groupby("source")["leak_ratio"].apply(lambda s: (s > 0.3).mean()).plot.bar(figsize=(8, 3), rot=0)
ax.set_title("Share of issues that leak >30% of the gold patch lines"); plt.show()

### 7.6 Read a few normalized samples (do this! heuristics lie)

In [ ]:
def show_task(row, max_chars=1500):
    print("=" * 110)
    print(f"[{row['source']}/{row['split']}] {row['instance_id']}  repo={row['repo_canon']}  lang={row['language']}")
    print(f"src_files={row['n_src_files']} src_lines={row['src_lines']} f2p={row['n_f2p']} env={row['env_available']} leak={row['leak_ratio']:.2f}"
          + (f"  score={row['score']:.1f}" if 'score' in row else ""))
    print("-" * 40, "PROBLEM STATEMENT")
    print(textwrap.shorten(row["problem_statement"], max_chars, placeholder=" ...") if len(row["problem_statement"]) > max_chars else row["problem_statement"])
    print("-" * 40, "PATCH (head)")
    print(row["patch"][:900])

for s in df["source"].unique():
    show_task(df[df["source"] == s].sample(1, random_state=1).iloc[0], max_chars=800)

## 8. Hard filters (funnel)

In [ ]:
filters = [
    ("problem statement too short",   df["ps_chars"] < MIN_PS_CHARS),
    ("problem statement too long",    df["ps_est_tokens"] > MAX_PS_TOKENS),
    ("no non-test source changes",    df["n_src_files"] == 0),
    ("patch touches no code files",   df["n_code_files"] == 0),
    ("too many files",                df["n_src_files"] > MAX_SRC_FILES),
    ("too many lines",                df["src_lines"] > MAX_SRC_LINES),
    ("issue leaks gold patch",        df["leak_ratio"] > MAX_LEAK_RATIO),
    ("mostly URL / non-English",      (df["ps_url_frac"] > 0.5) | (df["ps_nonascii"] > 0.3)),
]
if DROP_UNLINKED_F2P:
    filters.append(("F2P unrelated to test_patch (mixed-up row)", ~df["f2p_linked"]))
if REQUIRE_ENV:
    filters.append(("no runnable environment", ~df["env_available"]))
if REQUIRE_TESTS:
    filters.append(("no verification tests", ~df["has_tests"]))
if LANGUAGES:
    filters.append(("language not selected", ~df["language"].isin(LANGUAGES)))

alive = pd.Series(True, index=df.index)
funnel = [("start", alive.sum())]
removed_by = pd.DataFrame(index=df["source"].unique())
for name, mask in filters:
    hit = alive & mask
    removed_by[name] = df[hit].groupby("source").size()
    alive &= ~mask
    funnel.append((name, alive.sum()))

display(pd.DataFrame(funnel, columns=["after filter", "rows left"]))
display(removed_by.fillna(0).astype(int).T)
pool = df[alive].copy()
print(f"surviving pool: {len(pool):,}")
pool.groupby("source").size().rename("rows")

## 9. Quality score
Score (0..100) = 95 × weighted mix of four components (each 0..1) × leak penalty, + 3 × tie-breaker, + 2 for hints:
- **prior** (20%): trust in the source
- **verifiability** (25%): runnable env + tests, sane number of FAIL_TO_PASS
- **problem statement** (30%): good length, code, traceback, repro steps, expected-vs-actual.
  Multiplied by `SYNTHETIC_PS_DISCOUNT` for LLM-written issues (SWE-smith, R2E-Gym), which match these signals by template.
- **patch shape** (25%): prefers 5-150 changed lines in 1-3 files; trivial or huge patches make poor SFT trajectories
- **tie-breaker**: continuous issue length and patch size, so tasks with identical components no longer tie on a few score values


In [ ]:
def ps_len_score(n):
    return 0.1 if n < 100 else 0.5 if n < 250 else 1.0 if n <= 3000 else 0.8 if n <= 6000 else 0.5

def lines_score(n):
    return 0.4 if n <= 2 else 0.8 if n <= 5 else 1.0 if n <= 150 else 0.75 if n <= 400 else 0.45 if n <= 1000 else 0.2

def files_score(n):
    return 1.0 if n <= 3 else 0.85 if n <= 6 else 0.6 if n <= 12 else 0.35

def score_df(d):
    d = d.copy()
    d["s_prior"] = [lookup(SOURCE_PRIOR, s, sp, 0.5) for s, sp in zip(d["source"], d["split"])]
    f2p_ok = d["n_f2p"].isna() | (d["n_f2p"] <= 200)
    d["s_verif"] = 0.5 * d["env_available"] + 0.5 * d["has_tests"] * np.where(f2p_ok, 1.0, 0.5)
    d["s_ps"] = (0.40 * d["ps_chars"].map(ps_len_score)
                 + 0.15 * d["ps_has_code"] + 0.15 * d["ps_has_trace"]
                 + 0.15 * d["ps_has_repro"] + 0.15 * d["ps_has_expect"])
    d["s_ps"] *= np.where(d["source"].isin(SYNTHETIC_ISSUE_SOURCES), SYNTHETIC_PS_DISCOUNT, 1.0)
    d["s_patch"] = d["src_lines"].map(lines_score) * d["n_src_files"].map(files_score)
    raw_score = 0.20 * d["s_prior"] + 0.25 * d["s_verif"] + 0.30 * d["s_ps"] + 0.25 * d["s_patch"]
    d["s_tiebreak"] = 0.5 * d["ps_chars"].clip(upper=3000) / 3000 + 0.5 * d["src_lines"].clip(upper=150) / 150
    d["score"] = 95 * raw_score * (1 - 0.7 * d["leak_ratio"]) + 3 * d["s_tiebreak"] + 2 * d["has_hints"]  # max 100
    return d

pool = score_df(pool)

fig, ax = plt.subplots(figsize=(12, 4))
for s in sorted(pool["source"].unique()):
    ax.hist(pool.loc[pool["source"] == s, "score"], bins=50, histtype="step", linewidth=1.6, label=s)
ax.set_title("Quality score by source"); ax.legend(fontsize=8); plt.show()
print(f"score range: {pool['score'].min():.1f} - {pool['score'].max():.1f}")
pool.groupby("source")[["s_prior", "s_verif", "s_ps", "s_patch", "s_tiebreak", "score"]].mean().round(3)

### 9.1 Sanity-check the extremes

In [ ]:
print("##### HIGHEST scored")
for _, r in pool.nlargest(3, "score").iterrows():
    show_task(r, max_chars=600)
print("\n\n##### LOWEST scored")
for _, r in pool.nsmallest(3, "score").iterrows():
    show_task(r, max_chars=600)

## 10. Deduplication
Order matters: rows are processed best-first, so whenever two rows collide the **higher-scored** one survives
(e.g. a SWE-bench test row that is also in Verified keeps the Verified copy).
1. exact: `instance_id`, normalized `patch_hash`, normalized `ps_hash`
2. near-duplicate problem statements with MinHash-LSH

In [ ]:
pool = pool.sort_values("score", ascending=False)
n0 = len(pool)
for key in ["instance_id", "patch_hash", "ps_hash"]:
    before = len(pool)
    pool = pool[~(pool[key].notna() & pool[key].duplicated(keep="first"))]
    print(f"exact dedup on {key:12s}: -{before - len(pool):,}")
print(f"after exact dedup: {len(pool):,} (from {n0:,})")

In [ ]:
from datasketch import MinHash, MinHashLSH

def shingles(text, k=5, max_chars=4000):
    toks = re.findall(r"\w+", text[:max_chars].lower())
    return {" ".join(toks[i:i + k]) for i in range(max(1, len(toks) - k + 1))}

lsh = MinHashLSH(threshold=NEAR_DUP_THRESHOLD, num_perm=NUM_PERM)
keep, dup_of = [], {}
for uid, ps in tqdm(zip(pool["uid"], pool["problem_statement"]), total=len(pool), desc="near-dup"):
    m = MinHash(num_perm=NUM_PERM)
    m.update_batch([s.encode() for s in shingles(ps)])
    hits = lsh.query(m)
    if hits:
        dup_of[uid] = hits[0]
        continue
    lsh.insert(uid, m)
    keep.append(uid)

near_dups = pool[pool["uid"].isin(dup_of)]
pool = pool[pool["uid"].isin(set(keep))]
print(f"near-duplicates removed: {len(near_dups):,}  ->  pool: {len(pool):,}")
display(near_dups.groupby("source").size().rename("near_dups_removed"))

In [ ]:
# eyeball a couple of near-dup pairs to make sure the threshold is sensible
by_uid = df.set_index("uid")
for u in list(dup_of)[:2]:
    a, b = by_uid.loc[u], by_uid.loc[dup_of[u]]
    print("=" * 110)
    print(f"DROPPED {u} ({a['repo_canon']})\n{a['problem_statement'][:500]}")
    print(f"\nKEPT    {dup_of[u]} ({b['repo_canon']})\n{b['problem_statement'][:500]}")

## 11. Diversity-aware selection
The primary set and the reserve are selected separately, so `SOURCE_MAX_FRAC` applies to the 10K you actually train on.
If the caps leave slots empty, the repo cap is lifted first; the source cap is only lifted as a last resort (and reported).


In [ ]:
def select_diverse(pool, n_total, max_per_repo, source_max_frac, take_all_sources, exclude=()):
    pool = pool[~pool["uid"].isin(set(exclude))].sort_values("score", ascending=False)
    src_cap = {s: int(f * n_total) for s, f in source_max_frac.items()}
    chosen, chosen_set = [], set()
    repo_ct, src_ct = Counter(), Counter()

    def try_add(r, use_repo_cap=True, use_src_cap=True):
        if r.uid in chosen_set or len(chosen) >= n_total:
            return False
        if use_repo_cap and repo_ct[r.repo_canon] >= max_per_repo:
            return False
        if use_src_cap and src_ct[r.source] >= src_cap.get(r.source, n_total):
            return False
        chosen.append(r.uid); chosen_set.add(r.uid)
        repo_ct[r.repo_canon] += 1; src_ct[r.source] += 1
        return True

    for r in pool[pool["source"].isin(take_all_sources)].itertuples():   # pass 0: small, high-value sets
        try_add(r)
    for r in pool.itertuples():                                           # pass 1: best-first, all caps
        try_add(r)
    for label, kw in [("repo cap lifted, source cap kept", dict(use_repo_cap=False)),
                      ("ALL caps lifted", dict(use_repo_cap=False, use_src_cap=False))]:
        if len(chosen) >= n_total:
            break
        added = Counter()
        for r in pool.itertuples():
            if len(chosen) >= n_total:
                break
            if try_add(r, **kw):
                added[r.source] += 1
        print(f"  {n_total - len(chosen) + sum(added.values()):,} slots were empty -> {label}: added {dict(added)}")

    out = pool.set_index("uid").loc[chosen].reset_index()
    out["selection_rank"] = np.arange(len(out))
    return out

print("primary:")
primary = select_diverse(pool, TARGET_N, MAX_PER_REPO, SOURCE_MAX_FRAC, TAKE_ALL_SOURCES)
print("reserve:")
reserve = select_diverse(pool, RESERVE_N, MAX_PER_REPO, SOURCE_MAX_FRAC, TAKE_ALL_SOURCES, exclude=primary["uid"])
reserve["selection_rank"] += len(primary)

assert not set(primary["uid"]) & set(reserve["uid"]), "primary and reserve overlap"
assert primary["instance_id"].is_unique
print(f"\nprimary: {len(primary):,} | reserve: {len(reserve):,}")
for s, f in SOURCE_MAX_FRAC.items():
    share = (primary["source"] == s).mean()
    print(f"  {s:<12s} {share:6.1%} of primary (cap {f:.0%}) {'OK' if share <= f + 1e-9 else 'OVER CAP'}")


## 12. Inspect the final set

In [ ]:
comp = pd.DataFrame({
    "pool":    pool["source"].value_counts(),
    "primary": primary["source"].value_counts(),
    "reserve": reserve["source"].value_counts(),
}).fillna(0).astype(int)
comp["primary_%"] = (comp["primary"] / comp["primary"].sum() * 100).round(1)
display(comp.sort_values("primary", ascending=False))

display(pd.DataFrame({
    "pool": pool["language"].value_counts(normalize=True).round(3),
    "primary": primary["language"].value_counts(normalize=True).round(3),
}).fillna(0).sort_values("primary", ascending=False).head(10))

print(f"repos in primary: {primary['repo_canon'].nunique():,}")
display(primary["repo_canon"].value_counts().head(15).rename("tasks"))
print("repos without an owner in primary:", sorted(r for r in primary["repo_canon"].unique() if "/" not in r) or "none")

In [ ]:
cmp = pd.concat([pool.assign(set="pool"), primary.assign(set="primary")])
for col, logx in [("score", False), ("ps_chars", True), ("src_lines", True), ("n_src_files", True)]:
    fig, ax = plt.subplots(figsize=(10, 3))
    for name, g in cmp.groupby("set"):
        v = g[col].astype(float)
        v = v[v >= 1] if logx else v
        bins = np.logspace(0, np.log10(v.max() + 1), 40) if logx else 40
        ax.hist(v, bins=bins, histtype="step", density=True, linewidth=1.8, label=name)
    if logx: ax.set_xscale("log")
    ax.set_title(f"{col}: pool vs selected"); ax.legend(); plt.show()

In [ ]:
for _, r in primary.sample(3, random_state=7).iterrows():
    show_task(r, max_chars=700)

### 12.1 Integrity check (stops here if anything is off)
Every selected row is rebuilt from its original dataset row with `normalize_row` and compared field by field.
Any mismatch means a column got out of step with its row somewhere in the pipeline, so the notebook raises before saving.
The second check confirms that no selected row has FAIL_TO_PASS tests unrelated to its `test_patch` (section 8 already drops those).

In [ ]:
CHECK_FIELDS = ["instance_id", "repo", "base_commit", "problem_statement", "patch", "setup_patch",
                "test_patch", "f2p", "p2p", "env_image", "version"]

def _norm(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return None
    if isinstance(v, np.ndarray):
        v = v.tolist()
    return v

def integrity_check(sel, name):
    bad = []
    for _, rec in tqdm(sel.iterrows(), total=len(sel), desc=f"integrity {name}", leave=False):
        fresh = normalize_row(rec["source"], rec["split"], int(rec["row_idx"]), original_row(rec))
        diff = [f for f in CHECK_FIELDS if _norm(fresh.get(f)) != _norm(rec.get(f))]
        if diff:
            bad.append((rec["uid"], diff))
    print(f"{name}: {len(sel):,} rows checked, {len(bad)} misaligned")
    for uid, diff in bad[:10]:
        print("   ", uid, "fields differ:", diff)
    return bad

bad_primary = integrity_check(primary, "primary")
bad_reserve = integrity_check(reserve, "reserve")
assert not bad_primary and not bad_reserve, "misaligned rows: see the list above"

for name, sel in [("primary", primary), ("reserve", reserve)]:
    unlinked = sel[[not f2p_in_test_patch(a, b) for a, b in zip(sel["f2p"], sel["test_patch"])]]
    print(f"{name}: {len(unlinked)} rows whose FAIL_TO_PASS never relate to test_patch (should be 0: section 8 drops them)")
    if len(unlinked):
        display(unlinked[["uid", "instance_id", "repo_canon"]].head(10))

### 12.2 Harness readiness: what the trajectory pipeline gets per source

In [ ]:
ready = primary.assign(
    has_image=primary["env_image"].notna(),
    has_version=primary["version"].notna(),
    has_setup_patch=primary["setup_patch"].fillna("").str.len() > 0,
    has_test_patch=primary["test_patch"].fillna("").str.len() > 0,
    has_f2p=primary["n_f2p"].fillna(0) > 0,
).groupby(["source", "verifier"]).agg(
    rows=("uid", "size"),
    image=("has_image", "mean"),
    image_derived=("env_image_source", lambda s: (s == "derived").mean()),
    version=("has_version", "mean"),
    test_patch=("has_test_patch", "mean"),
    f2p=("has_f2p", "mean"),
    setup_patch=("has_setup_patch", "mean"),
)
display(ready.style.format({c: "{:.0%}" for c in ready.columns if c != "rows"}))
print("SWE-bench-verified rows need `version` to pick their test command;"
      " SWE-smith needs setup_patch; R2E-Gym has no test_patch by design.")

## 13. Save

In [ ]:
KEEP = ["selection_rank", "uid", "source", "split", "row_idx", "instance_id", "repo", "repo_canon", "language",
        "base_commit", "fix_commit", "version", "environment_setup_commit",
        "problem_statement", "extra_context", "hints", "patch", "setup_patch", "test_patch", "commit_test_patch",
        "f2p", "p2p", "env_image", "env_image_source", "env_available", "verifier", "created_at",
        "n_src_files", "src_lines", "n_f2p", "n_tests",
        "ps_chars", "leak_ratio", "s_prior", "s_verif", "s_ps", "s_patch", "s_tiebreak", "score", "raw_json"]
# NOTE: `hints` is saved for reference only. Do not put it in the agent prompt (it can describe the fix).

def original_row(rec):
    return raw[rec["source"]][rec["split"]][int(rec["row_idx"])]

def raw_json(rec):
    """The original dataset row (minus very large fields), so each source's harness fields survive."""
    o, dropped = {}, []
    for k, v in original_row(rec).items():
        if k in RAW_DROP_KEYS or (isinstance(v, str) and len(v) > RAW_MAX_FIELD_CHARS):
            dropped.append(k)
            continue
        o[k] = v
    o["_dropped_keys"] = dropped
    return json.dumps(o, default=str, ensure_ascii=False)

def to_save(d):
    d = d.copy()
    d["raw_json"] = [raw_json(r) for _, r in tqdm(d.iterrows(), total=len(d), desc="raw_json", leave=False)]
    d = d[KEEP]
    for c in ["f2p", "p2p"]:
        d[c] = d[c].map(lambda x: json.dumps(list(x)) if isinstance(x, (list, np.ndarray)) else None)
    return d

primary_out, reserve_out = to_save(primary), to_save(reserve)
primary_out.to_parquet("sft_tasks_primary_10k.parquet", index=False)
reserve_out.to_parquet("sft_tasks_reserve.parquet", index=False)
primary_out.to_json("sft_tasks_primary_10k.jsonl", orient="records", lines=True)
print("saved:", "sft_tasks_primary_10k.parquet", "sft_tasks_reserve.parquet", "sft_tasks_primary_10k.jsonl")

# from google.colab import files; files.download("sft_tasks_primary_10k.parquet")
# from datasets import Dataset; Dataset.from_pandas(primary_out).push_to_hub("YOUR_USER/swe-sft-tasks-10k", private=True)

In [ ]:
# Get the original row back (with every harness-specific field) for any selected task
_ = original_row(primary.iloc[0]); print(list(_.keys()))
print(json.loads(primary_out.iloc[0]["raw_json"]).keys())

## 14. Copy to Google Drive (Colab)

In [ ]:
import os, shutil
from google.colab import drive
drive.mount("/content/drive")

OUT_DIR = "/content/drive/MyDrive/Khalil_Hennara/codeAgentPandasData"
HF_DIR  = "/content/drive/MyDrive/Khalil_Hennara/codeAgentHF"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(HF_DIR, exist_ok=True)
for f in ["sft_tasks_primary_10k.parquet", "sft_tasks_reserve.parquet", "sft_tasks_primary_10k.jsonl"]:
    shutil.copy(f, OUT_DIR)

from datasets import Dataset
Dataset.from_pandas(primary_out, preserve_index=False).save_to_disk(f"{HF_DIR}/primary")
Dataset.from_pandas(reserve_out, preserve_index=False).save_to_disk(f"{HF_DIR}/reserve")
print("copied to", OUT_DIR, "and", HF_DIR)